# **🤗 Contradictory, My Dear Watson ([캐글 경진대회](https://www.kaggle.com/competitions/contradictory-my-dear-watson?rvi=1))**

## **1. 라이브러리 가져오기**

In [ ]:
!pip install evaluate
!pip install wandb

import torch
import numpy as np

# 2. 데이터셋 불러오기 (.csv 파일 I/O)
import pandas as pd

# 3. 데이터 시각화
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns

# 4. 데이터 전처리
from datasets import Dataset, DatasetDict
import evaluate
from sklearn.model_selection import train_test_split

# 5. 모델 구축
import torch.nn as nn
import evaluate
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, TrainingArguments, Trainer, XLMRobertaModel

# 6. 모델 훈련
from sklearn.metrics import accuracy_score, f1_score
from transformers import Trainer, EarlyStoppingCallback

In [ ]:
# GPU 환경 확인 (PyTorch)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 환경 변수 설정
import os
os.environ["WANDB_API_KEY"] = "0"

In [ ]:
# 버전 에러 방지
import warnings
warnings.filterwarnings("ignore")

! pip install -U accelerate
! pip install -U transformers

## **2. 데이터셋 불러오기**

In [ ]:
train_data = pd.read_csv("../data/train.csv")
test_data = pd.read_csv("../data/test.csv")

In [ ]:
# 데이터셋 구조 확인
display(train_data.info())
display(train_data.head())

display(test_data.info())
display(test_data.head())

## **3. 데이터 시각화**

In [ ]:
# train_data language 분포
labels, frequencies = np.unique(train_data.language.values, return_counts=True)
fig1 = px.pie(values=frequencies, names=labels, title="train_data language 분포", color_discrete_sequence=px.colors.sequential.Plotly3)
fig1.show()

# test_data language 분포
labels, frequencies = np.unique(test_data.language.values, return_counts=True)
fig2 = px.pie(values=frequencies, names=labels, title="test_data language 분포", color_discrete_sequence=px.colors.sequential.Plotly3)
fig2.show()

In [ ]:
# train_data label 분포
label_count = train_data["label"].value_counts().sort_index()
label_names = ["수반(0)","중립(1)","모순(2)"]
label_count.index = label_names
fig3 = go.Figure([go.Bar(x=label_names, y=label_count, marker_color="skyblue")])
fig3.update_layout(title_text="train_data label 분포", xaxis_title_text="label", yaxis_title_text="count")
fig3.show()

In [ ]:
# train_data premise 데이터 길이 분포
train_data["premise_text_length"] = train_data["premise"].apply(len)
sns.histplot(train_data,x="premise_text_length", kde=True)

In [ ]:
# train_data hypothesis 데이터 길이 분포
train_data["hypothesis_text_length"] = train_data["hypothesis"].apply(len)
sns.histplot(train_data,x="hypothesis_text_length", kde=True)

## **4. 데이터 전처리**

In [ ]:
# 불필요한 열 제거
train_data = train_data.drop(labels=["language","lang_abv","premise_text_length","hypothesis_text_length"], axis=1)
test_data = test_data.drop(labels=["language","lang_abv"], axis=1)

In [ ]:
# 데이터 분할 및 변환 (과적합 방지, 일반화 개선)
train_df, val_df = train_test_split(train_data, test_size=0.25, random_state=201818786)
train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)
test_ds = Dataset.from_pandas(test_data)

ds = DatasetDict()
ds["train"] = train_ds
ds["validation"] = val_ds
ds["test"] = test_ds

## **5. 모델 구축 ([XLM-RoBERTa 사용](https://huggingface.co/symanto/xlm-roberta-base-snli-mnli-anli-xnli))**

In [ ]:
# 토크나이저 불러오기
model_name = "symanto/xlm-roberta-base-snli-mnli-anli-xnli"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
# 토크나이징 수행 함수
def tokenizer_sentence(data):
    return tokenizer(data["premise"], data["hypothesis"], truncation=True)

In [ ]:
# 데이터 토큰화
tokenized_ds = ds.map(tokenizer_sentence, batched=True)

In [ ]:
# 데이터 패딩 처리
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
# 모델 설계
class CustomXLMRobertaModel(nn.Module):
    def __init__(self, num_labels):
        super(CustomXLMRobertaModel, self).__init__()
        model_name = "symanto/xlm-roberta-base-snli-mnli-anli-xnli"
        self.roberta = XLMRobertaModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.25)
        self.classifier = nn.Sequential(nn.Linear(768,512), nn.LayerNorm(512), nn.ReLU(), nn.Dropout(0.25), nn.Linear(512, num_labels))
        self.loss = nn.CrossEntropyLoss()
        self.num_labels = num_labels

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        output = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        output = self.dropout(output.last_hidden_state[:, 0, :])
        logits = self.classifier(output)

        if labels is not None:
            loss = self.loss(logits.view(-1, self.num_labels), labels.view(-1))
            return {"loss": loss, "logits": logits}
        else:
            return logits

In [ ]:
# 클래스 인스턴스 생성 및 모델에 할당
model = CustomXLMRobertaModel(num_labels=3)

## **6. 모델 훈련 ([Trainer 사용](https://huggingface.co/docs/transformers/main_classes/trainer))([Early Stopping 사용](https://huggingface.co/docs/transformers/main_classes/callback))**

In [ ]:
# TrainingArguments 인스턴스 생성 및 평가 지표 함수 생성
training_args = TrainingArguments("./outputs", optim="adamw_torch", num_train_epochs=5, eval_strategy="epoch",
                                  logging_dir='./logs', logging_steps=500, report_to="none")
# load_best_model_at_end=True

f1_metric = evaluate.load("f1")

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_metric.compute(predictions=predictions, references=labels, average="micro")["f1"]
    }

In [ ]:
# Trainer 클래스 정의 (Early Stopping 콜백 함수 사용 → 과적합 방지)
trainer = Trainer(
    model, args=training_args, train_dataset=tokenized_ds["train"], eval_dataset=tokenized_ds["validation"], data_collator=data_collator,
    tokenizer=tokenizer, compute_metrics=compute_metrics
)
# callbacks=[EarlyStoppingCallback(3)]

In [ ]:
# 환경 변수 설정
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
# Trainer 클래스를 사용한 모델 훈련
trainer.train()

## **7. 모델 예측 및 제출**

In [ ]:
# 모델 예측 수행
predictions = trainer.predict(tokenized_ds["test"])

In [ ]:
# 로짓 값 확률로 변환
logits = torch.from_numpy(predictions.predictions)
probs = torch.softmax(logits, -1).tolist()

In [ ]:
# 예측 결과 포맷 맞추기
outputs = [(ds["test"]["id"][index], prob.index(max(prob))) for index, prob in enumerate(probs)]

In [ ]:
# 제출용 파일 생성
submission = pd.read_csv("../data/sample_submission.csv")
submission["prediction"] = pd.DataFrame(outputs)[1]
submission.to_csv("submission.csv", index=False)